# GNN-Pruning — overnight single-seed sweep (Colab / A100)

Runs the full pruning matrix on CUDA. The large undirected graphs (Reddit,
ogbn-products, Yelp) use a **sparse-adjacency (SpMM)** path so they fit in
GPU memory; everything else runs full-batch dense. Wanda scores are **exact**
(the sparse path is numerically identical to the dense one — regression-tested).

**Before running:**
1. `Runtime → Change runtime type → A100 GPU` (Colab Pro). L4 (24 GB) also works
   for everything except possibly ogbn-products.
2. Colab Pro: enable **background execution** so the sweep survives tab close.
3. Run the cells top to bottom. The sweep is **idempotent** — if you disconnect,
   just re-run the setup + sweep cells and it resumes (finished cells are skipped).

**Known infeasible cell:** `reddit / gat`. Full-batch GAT on 23M edges needs
~188 GB (attention is inherently per-edge); it will OOM and be logged as a
failure in `run.log`. Reddit is still covered by GCN and GraphSAGE. Everything
else is expected to complete.


## 1. Confirm GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No CUDA GPU — set Runtime → A100 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("torch:", torch.__version__)

## 2. Mount Drive (so results + datasets survive disconnects)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/gnn-pruning'   # change if you like
os.makedirs(DRIVE_ROOT + '/data', exist_ok=True)
os.makedirs(DRIVE_ROOT + '/results', exist_ok=True)
print('Persisting data/ and results/ under', DRIVE_ROOT)

## 3. Clone the branch + install deps
No `torch_sparse` needed — the sparse path uses native `torch.sparse`.

In [ ]:
%cd /content
![ -d GNN-Pruning-Research ] || git clone --branch main https://github.com/Mike-Mans/GNN-Pruning-Research.git
%cd /content/GNN-Pruning-Research
!git fetch origin && git checkout main && git pull --ff-only

In [ ]:
# Symlink data/ and results/ to Drive so they persist across sessions.
import os, shutil
for d in ['data', 'results']:
    if os.path.islink(d):
        continue
    if os.path.exists(d):
        shutil.rmtree(d)
    os.symlink(f'{DRIVE_ROOT}/{d}', d)
!ls -la data results

In [ ]:
# Colab ships torch+CUDA. Add PyG + project deps (no torch_sparse needed).
!pip -q install torch_geometric ogb rdkit
!pip -q install -e .
print('install done')

## 4. Run the sweep (no-pruning FIRST — pruning loads its checkpoints)
Large datasets are auto-ordered last. Expect a few hours on A100. `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` reduces fragmentation.

In [ ]:
import os, subprocess, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

CONFIGS = {
    'no-pruning':      'src/gnn_pruning/configs/no_pruning.yaml',
    'magnitude':       'src/gnn_pruning/configs/magnitude.yaml',
    'wanda-uniform':   'src/gnn_pruning/configs/wanda_uniform.yaml',
    'wanda-degree':    'src/gnn_pruning/configs/wanda_degree.yaml',
    'wanda-per-class': 'src/gnn_pruning/configs/wanda_per_class.yaml',
}
for method, cfg in CONFIGS.items():
    print(f'\n===== {method} =====', flush=True)
    rc = subprocess.run(
        [sys.executable, '-m', 'gnn_pruning.cli', 'sweep',
         '--method', method, '--config', cfg],
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    ).returncode
    print(f'{method} finished (rc={rc})', flush=True)
print('\nSWEEP COMPLETE')

## 5. Inspect results

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob('results/*/summary.csv')):
    df = pd.read_csv(f)
    print('\n=== ', f, ' (rows:', len(df), ') ===')
    display(df.head(12))

print('\n--- failures / OOMs across run logs (expect reddit/gat) ---')
!grep -H -A1 'FAILED' results/*/run.log | head -40

## Notes
- **Resuming after a disconnect:** re-run cells 1–4. Idempotency skips every
  `(dataset, arch, seed, split)` already on Drive; only unfinished cells run.
- **Multi-seed pass:** edit each `src/gnn_pruning/configs/*.yaml` to
  `seeds: [0, 1, 2, 3, 4]`, push, and re-run cell 4. Finished seed-0 cells are
  skipped; seeds 1–4 fill in. (Large-NC × 5 is the slow part — run selectively.)
- **Outputs** live in `results/<method>/<dataset>/<arch>/seed-<N>/split-<M>/`
  on your Drive; `summary.csv` has the mean over seeds × splits with `n_runs`.
